In [1]:
# ============================================================
# GANTT CHART DATASET ANALYSIS USING NLP + K-MEANS
# ============================================================

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import re
import warnings
warnings.filterwarnings("ignore")

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Machine Learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

# Download NLP resources
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

print("All libraries loaded successfully.")


# ============================================================
# 2. LOAD DATASET
# ============================================================

file_path = "GanttChart.csv"

try:
    df = pd.read_csv(file_path)
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print("ERROR: GanttChart.csv was not found.")
    print("Make sure GanttChart.csv is in the same folder as this notebook.")
    raise

print("\nDataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 10 rows:")
display(df.head(10))


# ============================================================
# 3. CLEAN COLUMN NAMES
# ============================================================

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
    .str.replace(r"[^\w]", "", regex=True)
)

print("\nCleaned column names:")
print(df.columns.tolist())


# ============================================================
# 4. DATASET INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

print("\nNumber of rows:", len(df))
print("Number of columns:", len(df.columns))

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())


# ============================================================
# 5. REMOVE DUPLICATES
# ============================================================

df = df.drop_duplicates().reset_index(drop=True)

print("\nDataset shape after removing duplicates:", df.shape)


# ============================================================
# 6. IDENTIFY TASK COLUMN
# ============================================================

possible_task_columns = [
    "task",
    "tasks",
    "task_name",
    "task_description",
    "description",
    "activity",
    "activity_name",
    "name",
    "title",
    "work",
    "phase"
]

task_column = None

for column in possible_task_columns:
    if column in df.columns:
        task_column = column
        break

# If no standard task column exists, find first text column
if task_column is None:

    text_columns = df.select_dtypes(
        include=["object"]
    ).columns.tolist()

    if len(text_columns) > 0:
        task_column = text_columns[0]

if task_column is None:
    raise ValueError(
        "No text/task column could be identified."
    )

print("\nTask column selected:")
print(task_column)


# ============================================================
# 7. IDENTIFY DATE COLUMNS
# ============================================================

possible_start_columns = [
    "start",
    "start_date",
    "startdate",
    "begin",
    "begin_date",
    "beginning"
]

possible_end_columns = [
    "end",
    "end_date",
    "enddate",
    "finish",
    "finish_date",
    "completion_date"
]

start_column = None
end_column = None

for column in possible_start_columns:
    if column in df.columns:
        start_column = column
        break

for column in possible_end_columns:
    if column in df.columns:
        end_column = column
        break

print("\nStart date column:", start_column)
print("End date column:", end_column)


# ============================================================
# 8. CONVERT DATE COLUMNS
# ============================================================

if start_column is not None:

    df[start_column] = pd.to_datetime(
        df[start_column],
        errors="coerce"
    )

if end_column is not None:

    df[end_column] = pd.to_datetime(
        df[end_column],
        errors="coerce"
    )

if start_column is not None and end_column is not None:

    df["duration_days"] = (
        df[end_column] - df[start_column]
    ).dt.days

    print("\nDate information:")
    display(
        df[
            [
                task_column,
                start_column,
                end_column,
                "duration_days"
            ]
        ].head(10)
    )


# ============================================================
# 9. NLP TEXT PREPROCESSING
# ============================================================

stop_words = set(stopwords.words("english"))

lemmatizer = WordNetLemmatizer()


def clean_text(text):

    # Convert to string
    text = str(text)

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+|https\S+",
        "",
        text
    )

    # Remove email addresses
    text = re.sub(
        r"\S+@\S+",
        "",
        text
    )

    # Remove numbers
    text = re.sub(
        r"\d+",
        " ",
        text
    )

    # Remove punctuation
    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    # Tokenize
    words = text.split()

    # Remove stopwords
    words = [
        word
        for word in words
        if word not in stop_words
    ]

    # Lemmatization
    words = [
        lemmatizer.lemmatize(word)
        for word in words
    ]

    return " ".join(words)


# Create cleaned text column
df["clean_task"] = (
    df[task_column]
    .fillna("")
    .apply(clean_text)
)

print("\nOriginal and cleaned task descriptions:")

display(
    df[
        [
            task_column,
            "clean_task"
        ]
    ].head(15)
)


# ============================================================
# 10. REMOVE EMPTY TEXT RECORDS
# ============================================================

df = df[
    df["clean_task"].str.strip() != ""
].reset_index(drop=True)

print(
    "\nDataset size after removing empty task descriptions:",
    df.shape
)


# ============================================================
# 11. TF-IDF VECTORIZATION
# ============================================================

tfidf = TfidfVectorizer(
    max_features=1000,
    min_df=1,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X = tfidf.fit_transform(
    df["clean_task"]
)

print("\nTF-IDF completed.")

print(
    "TF-IDF matrix shape:",
    X.shape
)


# ============================================================
# 12. DISPLAY IMPORTANT TF-IDF WORDS
# ============================================================

feature_names = tfidf.get_feature_names_out()

tfidf_scores = np.asarray(
    X.mean(axis=0)
).flatten()

tfidf_ranking = (
    pd.DataFrame({
        "word": feature_names,
        "tfidf_score": tfidf_scores
    })
    .sort_values(
        "tfidf_score",
        ascending=False
    )
)

print("\nTop 30 TF-IDF features:")

display(
    tfidf_ranking.head(30)
)


# ============================================================
# 13. FIND OPTIMAL NUMBER OF CLUSTERS
# ============================================================

if len(df) < 3:

    raise ValueError(
        "At least 3 task records are required for clustering."
    )

max_clusters = min(
    10,
    len(df) - 1
)

cluster_values = range(
    2,
    max_clusters + 1
)

inertia_values = []
silhouette_values = []

for k in cluster_values:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(X)

    inertia_values.append(
        model.inertia_
    )

    silhouette_values.append(
        silhouette_score(
            X,
            labels
        )
    )


# ============================================================
# 14. ELBOW METHOD
# ============================================================

plt.figure(
    figsize=(10, 6)
)

plt.plot(
    list(cluster_values),
    inertia_values,
    marker="o"
)

plt.xlabel(
    "Number of Clusters"
)

plt.ylabel(
    "Inertia"
)

plt.title(
    "Elbow Method for Choosing K"
)

plt.xticks(
    list(cluster_values)
)

plt.grid(
    True,
    alpha=0.3
)

plt.show()


# ============================================================
# 15. SILHOUETTE SCORE
# ============================================================

plt.figure(
    figsize=(10, 6)
)

plt.plot(
    list(cluster_values),
    silhouette_values,
    marker="o"
)

plt.xlabel(
    "Number of Clusters"
)

plt.ylabel(
    "Silhouette Score"
)

plt.title(
    "Silhouette Score for Different K Values"
)

plt.xticks(
    list(cluster_values)
)

plt.grid(
    True,
    alpha=0.3
)

plt.show()


# ============================================================
# 16. SELECT BEST K
# ============================================================

best_index = np.argmax(
    silhouette_values
)

best_k = list(
    cluster_values
)[best_index]

best_silhouette = silhouette_values[
    best_index
]

print(
    "\nRecommended number of clusters:",
    best_k
)

print(
    "Best silhouette score:",
    round(
        best_silhouette,
        4
    )
)


# ============================================================
# 17. APPLY K-MEANS
# ============================================================

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=20
)

df["cluster"] = kmeans.fit_predict(
    X
)

print(
    "\nK-Means clustering completed."
)


# ============================================================
# 18. DISPLAY CLUSTERED DATA
# ============================================================

print("\nClustered tasks:")

display(
    df[
        [
            task_column,
            "clean_task",
            "cluster"
        ]
    ].sort_values(
        "cluster"
    )
)


# ============================================================
# 19. CLUSTER SIZE
# ============================================================

cluster_counts = (
    df["cluster"]
    .value_counts()
    .sort_index()
)

print(
    "\nNumber of tasks in each cluster:"
)

display(
    cluster_counts
)


# ============================================================
# 20. CLUSTER SIZE VISUALISATION
# ============================================================

plt.figure(
    figsize=(10, 6)
)

plt.bar(
    cluster_counts.index.astype(str),
    cluster_counts.values
)

plt.xlabel(
    "Cluster"
)

plt.ylabel(
    "Number of Tasks"
)

plt.title(
    "Number of Tasks in Each Cluster"
)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.show()


# ============================================================
# 21. MOST IMPORTANT WORDS FOR EACH CLUSTER
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "IMPORTANT WORDS FOR EACH CLUSTER"
)

print(
    "=" * 70
)

cluster_keywords = {}

for cluster_number in range(best_k):

    cluster_center = (
        kmeans.cluster_centers_[
            cluster_number
        ]
    )

    top_indices = (
        cluster_center
        .argsort()[::-1][:15]
    )

    top_words = [
        feature_names[index]
        for index in top_indices
    ]

    cluster_keywords[
        cluster_number
    ] = top_words

    print(
        f"\nCluster {cluster_number}:"
    )

    print(
        ", ".join(top_words)
    )


# ============================================================
# 22. SHOW TASKS BY CLUSTER
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "TASKS GROUPED BY CLUSTER"
)

print(
    "=" * 70
)

for cluster_number in range(best_k):

    print(
        f"\nCLUSTER {cluster_number}"
    )

    print(
        "-" * 50
    )

    cluster_tasks = df[
        df["cluster"] == cluster_number
    ][task_column]

    for task in cluster_tasks:

        print(
            "•",
            task
        )


# ============================================================
# 23. CLUSTER QUALITY METRICS
# ============================================================

cluster_labels = df["cluster"]

silhouette = silhouette_score(
    X,
    cluster_labels
)

davies_bouldin = davies_bouldin_score(
    X.toarray(),
    cluster_labels
)

calinski = calinski_harabasz_score(
    X.toarray(),
    cluster_labels
)

print(
    "\n" + "=" * 70
)

print(
    "CLUSTER QUALITY METRICS"
)

print(
    "=" * 70
)

print(
    "Silhouette Score:",
    round(silhouette, 4)
)

print(
    "Davies-Bouldin Index:",
    round(davies_bouldin, 4)
)

print(
    "Calinski-Harabasz Score:",
    round(calinski, 4)
)


# ============================================================
# 24. PCA VISUALISATION
# ============================================================

# Convert TF-IDF matrix to dense matrix
X_dense = X.toarray()

pca = PCA(
    n_components=2,
    random_state=42
)

X_pca = pca.fit_transform(
    X_dense
)

df["pca_1"] = X_pca[:, 0]
df["pca_2"] = X_pca[:, 1]


# ============================================================
# 25. PLOT K-MEANS CLUSTERS
# ============================================================

plt.figure(
    figsize=(12, 8)
)

scatter = plt.scatter(
    df["pca_1"],
    df["pca_2"],
    c=df["cluster"],
    cmap="viridis",
    s=100,
    alpha=0.8
)

plt.xlabel(
    "PCA Component 1"
)

plt.ylabel(
    "PCA Component 2"
)

plt.title(
    "K-Means Clustering of Gantt Tasks Using NLP"
)

plt.colorbar(
    scatter,
    label="Cluster"
)

plt.grid(
    True,
    alpha=0.2
)

plt.show()


# ============================================================
# 26. PCA VISUALISATION WITH TASK NAMES
# ============================================================

plt.figure(
    figsize=(16, 10)
)

plt.scatter(
    df["pca_1"],
    df["pca_2"],
    c=df["cluster"],
    cmap="viridis",
    s=100,
    alpha=0.8
)

for i in range(len(df)):

    task_name = str(
        df.iloc[i][task_column]
    )

    # Limit label length
    if len(task_name) > 40:
        task_name = (
            task_name[:40] + "..."
        )

    plt.annotate(
        task_name,
        (
            df.iloc[i]["pca_1"],
            df.iloc[i]["pca_2"]
        ),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8
    )

plt.xlabel(
    "PCA Component 1"
)

plt.ylabel(
    "PCA Component 2"
)

plt.title(
    "Gantt Tasks Grouped by NLP + K-Means"
)

plt.grid(
    True,
    alpha=0.2
)

plt.show()


# ============================================================
# 27. DURATION ANALYSIS
# ============================================================

if (
    start_column is not None
    and end_column is not None
):

    print(
        "\n" + "=" * 70
    )

    print(
        "TASK DURATION ANALYSIS"
    )

    print(
        "=" * 70
    )

    duration_summary = (
        df.groupby(
            "cluster"
        )["duration_days"]
        .agg(
            [
                "count",
                "mean",
                "median",
                "min",
                "max"
            ]
        )
        .round(2)
    )

    display(
        duration_summary
    )


# ============================================================
# 28. AVERAGE DURATION BY CLUSTER
# ============================================================

if "duration_days" in df.columns:

    plt.figure(
        figsize=(10, 6)
    )

    average_duration = (
        df.groupby(
            "cluster"
        )["duration_days"]
        .mean()
    )

    plt.bar(
        average_duration.index.astype(str),
        average_duration.values
    )

    plt.xlabel(
        "Cluster"
    )

    plt.ylabel(
        "Average Duration (Days)"
    )

    plt.title(
        "Average Task Duration by Cluster"
    )

    plt.grid(
        axis="y",
        alpha=0.3
    )

    plt.show()


# ============================================================
# 29. GANTT CHART VISUALISATION
# ============================================================

if (
    start_column is not None
    and end_column is not None
):

    gantt_df = df.dropna(
        subset=[
            start_column,
            end_column
        ]
    ).copy()

    gantt_df = gantt_df.sort_values(
        start_column
    ).reset_index(
        drop=True
    )

    plt.figure(
        figsize=(14, 10)
    )

    for i, row in gantt_df.iterrows():

        start_date = row[
            start_column
        ]

        end_date = row[
            end_column
        ]

        duration = (
            end_date - start_date
        ).days

        plt.barh(
            i,
            duration,
            left=start_date.toordinal(),
            alpha=0.8
        )

        plt.text(
            start_date.toordinal(),
            i,
            "  " + str(
                row[task_column]
            )[:35],
            va="center",
            fontsize=8
        )

    plt.yticks([])

    plt.xlabel(
        "Date"
    )

    plt.title(
        "Gantt Chart of Project Tasks"
    )

    # Convert x-axis ordinal values to dates
    ax = plt.gca()

    xmin, xmax = ax.get_xlim()

    dates = pd.date_range(
        start=min(
            gantt_df[start_column]
        ),
        end=max(
            gantt_df[end_column]
        ),
        periods=8
    )

    ax.set_xticks(
        [
            date.toordinal()
            for date in dates
        ]
    )

    ax.set_xticklabels(
        [
            date.strftime("%Y-%m-%d")
            for date in dates
        ],
        rotation=45
    )

    plt.grid(
        axis="x",
        alpha=0.3
    )

    plt.tight_layout()

    plt.show()


# ============================================================
# 30. CLUSTER + DATE ANALYSIS
# ============================================================

if (
    start_column is not None
    and end_column is not None
):

    cluster_date_summary = (
        df.groupby("cluster")
        .agg(
            start_date=(
                start_column,
                "min"
            ),
            end_date=(
                end_column,
                "max"
            ),
            average_duration=(
                "duration_days",
                "mean"
            ),
            number_of_tasks=(
                task_column,
                "count"
            )
        )
        .round(2)
    )

    print(
        "\nCluster project timeline:"
    )

    display(
        cluster_date_summary
    )


# ============================================================
# 31. CREATE CLUSTER SUMMARY TABLE
# ============================================================

summary_rows = []

for cluster_number in range(best_k):

    cluster_data = df[
        df["cluster"] == cluster_number
    ]

    keywords = ", ".join(
        cluster_keywords[
            cluster_number
        ][:10]
    )

    row = {
        "cluster": cluster_number,
        "number_of_tasks": len(
            cluster_data
        ),
        "important_keywords": keywords
    }

    if "duration_days" in df.columns:

        row[
            "average_duration_days"
        ] = round(
            cluster_data[
                "duration_days"
            ].mean(),
            2
        )

    summary_rows.append(row)


cluster_summary = pd.DataFrame(
    summary_rows
)

print(
    "\n" + "=" * 70
)

print(
    "FINAL CLUSTER SUMMARY"
)

print(
    "=" * 70
)

display(
    cluster_summary
)


# ============================================================
# 32. SAVE CLUSTERED DATASET
# ============================================================

output_file = (
    "GanttChart_clustered.csv"
)

# Remove PCA columns if you don't want them in final output
final_df = df.copy()

final_df.to_csv(
    output_file,
    index=False
)

print(
    "\nClustered dataset saved successfully:"
)

print(
    output_file
)


# ============================================================
# 33. SAVE CLUSTER SUMMARY
# ============================================================

summary_file = (
    "GanttChart_cluster_summary.csv"
)

cluster_summary.to_csv(
    summary_file,
    index=False
)

print(
    "Cluster summary saved successfully:"
)

print(
    summary_file
)


# ============================================================
# 34. FINAL RESULTS
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "FINAL RESULTS"
)

print(
    "=" * 70
)

print(
    "Original number of records:",
    len(df)
)

print(
    "Number of clusters:",
    best_k
)

print(
    "Silhouette Score:",
    round(
        silhouette,
        4
    )
)

print(
    "\nTasks per cluster:"
)

print(
    cluster_counts
)

print(
    "\nOutput files:"
)

print(
    "1. GanttChart_clustered.csv"
)

print(
    "2. GanttChart_cluster_summary.csv"
)

print(
    "\nAnalysis completed successfully."
)

All libraries loaded successfully.
Dataset loaded successfully.

Dataset shape: (8, 4)

Column names:
['Task', 'Start', 'Duration', 'Resource']

First 10 rows:


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


,Task,Start,Duration,Resource
0,Task 1,1/1/2016,50,A
1,Task 2,2/20/2016,25,B
2,Task 3,1/1/2016,100,C
3,Task 4,4/10/2016,60,C
4,Task 5,6/9/2016,30,C
5,Task 6,4/10/2016,150,A
6,Task 7,9/7/2016,80,B
7,Task 8,11/26/2016,10,B



Cleaned column names:
['task', 'start', 'duration', 'resource']

DATASET INFORMATION

Number of rows: 8
Number of columns: 4

Data types:
task          str
start         str
duration    int64
resource      str
dtype: object

Missing values:


task        0
start       0
duration    0
resource    0
dtype: int64


Duplicate rows: 0

Dataset shape after removing duplicates: (8, 4)

Task column selected:
task

Start date column: start
End date column: None

Original and cleaned task descriptions:


,task,clean_task
0,Task 1,task
1,Task 2,task
2,Task 3,task
3,Task 4,task
4,Task 5,task
5,Task 6,task
6,Task 7,task
7,Task 8,task



Dataset size after removing empty task descriptions: (8, 5)


ValueError: After pruning, no terms remain. Try a lower min_df or a higher max_df.